# RetailOps AI
## Notebook 03 — Agent and Tools

This notebook builds the core RetailOps AI agent and connects it to retail data and policy tools.

The agent can:
- identify customers,
- inspect customer and order information,
- inspect product information,
- retrieve relevant retail policy,
- combine multiple tool calls to answer multi-step customer requests.

Write actions, MCP integration, and human approval are added in later notebooks.

Cell 2 — Setup

In [ ]:
!pip -q install openai rank-bm25

In [ ]:
from pathlib import Path
import subprocess
import json
import os
import re

REPO_DIR = Path("/content/tau2-bench")

if not REPO_DIR.exists():
    subprocess.run(
        [
            "git", "clone", "-q",
            "https://github.com/sierra-research/tau2-bench.git",
            str(REPO_DIR)
        ],
        check=True
    )

RETAIL_DIR = REPO_DIR / "data" / "tau2" / "domains" / "retail"

DB_PATH = RETAIL_DIR / "db.json"
POLICY_PATH = RETAIL_DIR / "policy.md"

with open(DB_PATH, "r", encoding="utf-8") as f:
    retail_db = json.load(f)

policy_text = POLICY_PATH.read_text(encoding="utf-8")

print(" Retail environment loaded.")
print("Users:", len(retail_db["users"]))
print("Orders:", len(retail_db["orders"]))
print("Products:", len(retail_db["products"]))

 Retail environment loaded.
Users: 500
Orders: 1000
Products: 50


Cell 3 — Policy Retriever

In [ ]:
from rank_bm25 import BM25Okapi

sections = []
current_title = "General"
current_lines = []

for line in policy_text.splitlines():

    if line.startswith("#"):

        if current_lines:
            sections.append({
                "title": current_title,
                "text": "\n".join(current_lines).strip()
            })

        current_title = line.lstrip("#").strip()
        current_lines = []

    else:
        current_lines.append(line)

if current_lines:
    sections.append({
        "title": current_title,
        "text": "\n".join(current_lines).strip()
    })

sections = [s for s in sections if s["text"]]


def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


corpus = [
    tokenize(section["title"] + " " + section["text"])
    for section in sections
]

bm25 = BM25Okapi(corpus)

print(" Policy retriever ready.")

 Policy retriever ready.


Cell 4 — Retail Tools

In [ ]:
def find_user_by_email(email: str):
    """Find a retail user using their email address."""

    email = email.strip().lower()

    for user_id, user in retail_db["users"].items():
        if user.get("email", "").lower() == email:
            return {
                "user_id": user_id,
                "name": user.get("name"),
                "orders": user.get("orders", [])
            }

    return {"error": "User not found"}


def get_user_details(user_id: str):
    """Return safe customer information and their order IDs."""

    user = retail_db["users"].get(user_id)

    if not user:
        return {"error": "User not found"}

    return {
        "user_id": user_id,
        "name": user.get("name"),
        "orders": user.get("orders", []),
        "payment_method_ids": list(
            user.get("payment_methods", {}).keys()
        )
    }


def get_order_details(order_id: str, user_id: str):
    """Return an order only if it belongs to the identified customer."""

    order = retail_db["orders"].get(order_id)

    if not order:
        return {"error": "Order not found"}

    if order.get("user_id") != user_id:
        return {"error": "Order does not belong to this customer"}

    return {
        "order_id": order_id,
        "user_id": order.get("user_id"),
        "status": order.get("status"),
        "items": order.get("items"),
        "payment_history": order.get("payment_history"),
        "cancel_reason": order.get("cancel_reason")
    }


def get_product_details(product_id: str):
    """Return product variants, availability and price."""

    product = retail_db["products"].get(product_id)

    if not product:
        return {"error": "Product not found"}

    return product


def retrieve_policy(query: str, top_k: int = 3):
    """Retrieve the most relevant retail policy sections."""

    scores = bm25.get_scores(tokenize(query))

    ranked = sorted(
        zip(sections, scores),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    return [
        {
            "title": section["title"],
            "text": section["text"],
            "score": round(float(score), 3)
        }
        for section, score in ranked
    ]


print("✅ 5 Retail tools created.")

✅ 5 Retail tools created.


Cell 5 — Tool Schemas

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "find_user_by_email",
            "description": "Find a retail customer by email.",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {"type": "string"}
                },
                "required": ["email"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "get_user_details",
            "description": "Get customer details and their order IDs.",
            "parameters": {
                "type": "object",
                "properties": {
                    "user_id": {"type": "string"}
                },
                "required": ["user_id"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "get_order_details",
            "description": (
                "Get an order only after the customer has been identified. "
                "The user_id must come from the customer lookup result."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string"
                    },
                    "user_id": {
                        "type": "string"
                    }
                },
                "required": [
                    "order_id",
                    "user_id"
                ]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "get_product_details",
            "description": "Get product variants, availability and price.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_id": {"type": "string"}
                },
                "required": ["product_id"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "retrieve_policy",
            "description": "Retrieve retail policy relevant to the customer request.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string"
                    },
                    "top_k": {
                        "type": "integer",
                        "default": 3
                    }
                },
                "required": ["query"]
            }
        }
    }
]

Cell 6 — Tool Executor

In [ ]:
TOOL_MAP = {
    "find_user_by_email": find_user_by_email,
    "get_user_details": get_user_details,
    "get_order_details": get_order_details,
    "get_product_details": get_product_details,
    "retrieve_policy": retrieve_policy
}


def execute_tool(name, arguments):

    if name not in TOOL_MAP:
        return {"error": f"Unknown tool: {name}"}

    return TOOL_MAP[name](**arguments)

Cell 7 — OpenRouter Key

In [ ]:
from google.colab import userdata
from openai import OpenAI

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

assert OPENROUTER_API_KEY, \
    "Add OPENROUTER_API_KEY to Colab Secrets first."

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

print("✅ OpenRouter connected.")

✅ OpenRouter connected.


Cell 8 — Agent Instructions

In [ ]:
SYSTEM_PROMPT = """
You are RetailOps AI, a retail customer-service agent.

Your job is to solve customer requests using the available retail tools
and the official retail policy.

Rules:

1. Use tools instead of guessing customer, order, product, or policy information.
2. Retrieve the relevant policy before making policy-dependent decisions.
3. Never invent an order status, product availability, customer detail, or policy rule.
4. Do not modify any retail data in this notebook.
5. If information is insufficient, clearly ask for the missing information.
6. Keep final answers concise and explain the relevant policy when needed.
7. Identify the customer before accessing an order.
8. Verify that the order belongs to the identified customer before discussing order details.
9. Use the user_id returned by the customer lookup tool when calling get_order_details.
10. Do not claim that an order is missing until get_order_details has been called with the correct user_id.
"""

Cell 9 — Agent Loop

In [ ]:
MODEL = "openai/gpt-4.1-mini"


def run_agent(user_request, max_steps=8):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_request
        }
    ]

    trace = []

    for step in range(max_steps):

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            parallel_tool_calls=False,
            max_tokens=1200
        )

        message = response.choices[0].message
        messages.append(message)

        if not message.tool_calls:

            return {
                "answer": message.content,
                "trace": trace,
                "steps": step + 1
            }

        for call in message.tool_calls:

            name = call.function.name

            arguments = json.loads(
                call.function.arguments
            )

            result = execute_tool(
                name,
                arguments
            )

            trace.append({
                "step": step + 1,
                "tool": name,
                "arguments": arguments,
                "result": result
            })

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(
                    result,
                    ensure_ascii=False,
                    default=str
                )
            })

    return {
        "answer": "Stopped after maximum steps.",
        "trace": trace,
        "steps": max_steps
    }

Cell 10 — Get a Real Example from the Dataset

In [ ]:
example_order_id, example_order = next(
    iter(retail_db["orders"].items())
)

example_user_id = example_order["user_id"]

example_user = retail_db["users"][example_user_id]

example_email = example_user["email"]

print("Example email:", example_email)
print("Example order:", example_order_id)
print("Order status:", example_order["status"])

Example email: james.li4495@example.com
Example order: #W2611340
Order status: processed


Cell 11 — Multi-Step Agent Demo

In [ ]:
example_order_id, example_order = next(
    iter(retail_db["orders"].items())
)

example_user_id = example_order["user_id"]

example_user = retail_db["users"][example_user_id]

example_email = example_user["email"]

print("Example email:", example_email)
print("Example order:", example_order_id)
print("Order status:", example_order["status"])

Example email: james.li4495@example.com
Example order: #W2611340
Order status: processed


In [ ]:
request = f"""
My email is {example_email}.

Please check my order {example_order_id},
tell me its current status,
and explain what the retail policy allows me to do with this order.
"""

result = run_agent(request)

print("FINAL ANSWER:\n")
print(result["answer"])

print("\nTOOLS USED:\n")

for step in result["trace"]:
    print(
        f"Step {step['step']} → {step['tool']}"
    )

FINAL ANSWER:

Your order #W2611340 is currently in "processed" status. According to the retail policy:

- You can only cancel or modify orders if their status is "pending," so cancellation or modification is not available for this processed order.
- Returns or exchanges are only allowed for orders with "delivered" status.
  
Let me know if you need any further help with this order.

TOOLS USED:

Step 1 → find_user_by_email
Step 2 → get_order_details
Step 3 → retrieve_policy


Cell 12 — Show Trace

In [ ]:
import pandas as pd

trace_df = pd.DataFrame([
    {
        "step": x["step"],
        "tool": x["tool"],
        "arguments": x["arguments"]
    }
    for x in result["trace"]
])

trace_df

,step,tool,arguments
0,1,find_user_by_email,{'email': 'james.li4495@example.com'}
1,2,get_order_details,"{'order_id': '#W2611340', 'user_id': 'james_li..."
2,3,retrieve_policy,{'query': 'order cancellation and return policy'}


## Conclusion

RetailOps AI can now perform multi-step retail reasoning using real tools connected to the τ²-Bench Retail environment.

The current agent can retrieve customer, order and product information and ground policy-dependent decisions using the retail policy retriever.

Write operations are intentionally excluded from this notebook.

### Next
Notebook 04 will add:
- MCP integration,
- write tools,
- explicit human approval before state-changing actions.